In [1]:
from transformers import pipeline
import torch

print(torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

2.10.0+cpu
GPU 사용 가능: False


In [2]:
# Zero-shot 분류 모델 로드
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=0 if torch.cuda.is_available() else -1
)

print("모델 로드 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

모델 로드 완료


In [3]:
# 성향/감정/관심사 라벨
candidate_labels = [
    # 분위기 / 성격
    "차분한 사람",
    "활발한 사람",
    "감성적인 사람",
    "이성적인 사람",
    "유머러스한 사람",
    "진지한 사람",
    "긍정적인 사람",
    "생각이 많은 사람",
    "낯을 가리는 사람",
    "자기표현이 많은 사람",

    # 사회성
    "외향적인 사람",
    "내향적인 사람",
    "사람 만나는 걸 좋아하는 사람",
    "혼자 있는 걸 좋아하는 사람",
    "깊은 관계를 중요하게 생각하는 사람",
    "편하게 대화하는 걸 좋아하는 사람",
    "공감 잘하는 사람",
    "리액션이 좋은 사람",

    # 감정 상태
    "외로운 상태",
    "불안한 상태",
    "스트레스 받은 상태",
    "편안한 상태",
    "지친 상태",
    "설레는 상태",
    "우울한 상태",
    "동기부여가 필요한 상태",

    # 관심사
    "음악 좋아하는 사람",
    "운동 좋아하는 사람",
    "공부 좋아하는 사람",
    "패션 관심 있는 사람",
    "화장품 관심 있는 사람",
    "게임 좋아하는 사람",
    "영화 좋아하는 사람",
    "드라마 좋아하는 사람",
    "요리 좋아하는 사람",
    "카페 좋아하는 사람",

    # 생활 스타일
    "자기관리 중요하게 생각하는 사람",
    "계획적으로 사는 사람",
    "즉흥적인 사람",
    "밤에 감성 타는 사람",
    "혼자 산책 좋아하는 사람",
    "집에서 쉬는 걸 좋아하는 사람",
    "새로운 경험 좋아하는 사람",

    # 관계 성향
    "다정한 사람",
    "배려심 있는 사람",
    "애정 표현이 많은 사람",
    "연락을 중요하게 생각하는 사람",
    "감정 공유를 중요하게 생각하는 사람"
]


In [5]:
# 사용자별 zero-shot 분석 결과 저장
user_traits = {}


def analyze_traits(text):

    result = classifier(
        text,
        candidate_labels,
        multi_label=True
    )

    labels = result["labels"]
    scores = result["scores"]

    trait_scores = []

    for label, score in zip(labels, scores):

        trait_scores.append({
            "label": label,
            "score": float(score)
        })

    # 점수 높은 순 정렬
    trait_scores.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # 상위 10개만 반환
    return trait_scores[:10]


while True:

    user_id = input("\n사용자 ID 입력 (q 입력 시 종료): ")

    if user_id == "q":
        break

    text = input("편지 입력: ")

    traits = analyze_traits(text)

    # 사용자별 저장
    user_traits[user_id] = traits

    print(f"\n[{user_id}] 성향 분석 완료")

    print("\n상위 5개 성향:")

    for item in traits[:5]:
        print(
            f"- {item['label']}: "
            f"{item['score']:.4f}"
        )


print("\n현재 user_traits:")
print(user_traits)


사용자 ID 입력 (q 입력 시 종료): 1
편지 입력: 요즘은 하루 끝나고 혼자 산책하는 시간이 제일 좋아. 이어폰 끼고 음악 들으면서 천천히 걷다 보면 생각도 정리되고 마음도 좀 편안해지는 느낌이 들더라. 시끄러운 분위기보다는 조용한 카페나 한적한 곳에서 시간 보내는 걸 더 좋아하는 편이야.

[1] 성향 분석 완료

상위 5개 성향:
- 음악 좋아하는 사람: 0.9987
- 편안한 상태: 0.9986
- 혼자 있는 걸 좋아하는 사람: 0.9985
- 혼자 산책 좋아하는 사람: 0.9979
- 외로운 상태: 0.9943

사용자 ID 입력 (q 입력 시 종료): 2
편지 입력: 최근에 헬스를 시작했는데 생각보다 운동하는 시간이 재밌더라. 몸 쓰고 나면 스트레스도 좀 풀리고 괜히 하루를 알차게 보낸 느낌이 들어서 좋아. 운동 끝나고 친구들이랑 맛있는 거 먹으면서 수다 떠는 것도 은근 큰 행복인 것 같아.최근에 헬스를 시작했는데 생각보다 운동하는 시간이 재밌더라. 몸 쓰고 나면 스트레스도 좀 풀리고 괜히 하루를 알차게 보낸 느낌이 들어서 좋아. 운동 끝나고 친구들이랑 맛있는 거 먹으면서 수다 떠는 것도 은근 큰 행복인 것 같아.

[2] 성향 분석 완료

상위 5개 성향:
- 운동 좋아하는 사람: 0.9960
- 설레는 상태: 0.9946
- 동기부여가 필요한 상태: 0.9894
- 편안한 상태: 0.9823
- 편하게 대화하는 걸 좋아하는 사람: 0.9809

사용자 ID 입력 (q 입력 시 종료): q
